# 001 — Anomaly Detection

Unsupervised anomaly detection comparison using four scikit-learn detectors:
**Isolation Forest**, **Local Outlier Factor (LOF)**, **One-Class SVM**, and
**Elliptic Envelope**.

Covers contamination tuning, ROC-AUC / PR-AUC evaluation (using injected
ground truth), 2-D decision boundary visualisation, and a ranked summary
of all methods.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained in this notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Data loading
feature_paths: list[str] = []          # local or gs:// URIs; empty -> synthetic
feature_cols: list[str] | None = None  # None -> use all columns
label_col: str | None = None           # ground-truth anomaly column (1=anomaly); None -> unsupervised

# Anomaly detection settings
contamination: float = 0.05
test_size: float = 0.3
random_state: int = 42
n_synthetic_samples: int = 2000        # number of inlier samples for synthetic data

# Outputs
metrics_json_path: str = "outputs/metrics/metrics.json"
plots_dir: str = "outputs/plots"
executed_notebook_path: str | None = None

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# Create output directories
for _d in [plots_dir, str(Path(metrics_json_path).parent), "outputs/runs"]:
    Path(_d).mkdir(parents=True, exist_ok=True)

RUN_START = datetime.now(timezone.utc)
print(f"Run started: {RUN_START.isoformat()}")
print(f"contamination={contamination}, test_size={test_size}, random_state={random_state}")

## 1 — Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Data loading / synthetic generation
# ---------------------------------------------------------------------------

if not feature_paths:
    print("No feature_paths provided — generating synthetic dataset.")
    rng = np.random.default_rng(random_state)

    # --- inliers: 3 Gaussian clusters ---
    n_inliers = n_synthetic_samples
    centers = [(-2.0, -2.0), (0.0, 2.0), (3.0, 0.0)]
    cov = [[0.4, 0.1], [0.1, 0.4]]
    per_cluster = n_inliers // 3
    remainder = n_inliers - per_cluster * 3

    inlier_chunks = []
    for i, center in enumerate(centers):
        n = per_cluster + (1 if i < remainder else 0)
        chunk = rng.multivariate_normal(mean=center, cov=cov, size=n)
        inlier_chunks.append(chunk)
    X_inliers = np.vstack(inlier_chunks)  # shape (n_inliers, 2)

    # --- outliers: uniform random in [-6, 6] range ---
    n_outliers = int(round(n_inliers * contamination / (1.0 - contamination)))
    X_outliers = rng.uniform(low=-6.0, high=6.0, size=(n_outliers, 2))

    X_all = np.vstack([X_inliers, X_outliers])
    y_all = np.array([0] * n_inliers + [1] * n_outliers, dtype=np.int32)

    feature_names = ["feature_0", "feature_1"]
    df = pl.DataFrame(
        {name: X_all[:, i].tolist() for i, name in enumerate(feature_names)}
    ).with_columns(pl.Series("is_anomaly", y_all))
    effective_label_col = "is_anomaly"
    print(f"Synthetic data: {n_inliers} inliers + {n_outliers} outliers = {len(df)} rows")

else:
    print(f"Loading {len(feature_paths)} file(s)...")
    frames = []
    for path in feature_paths:
        if path.endswith(".parquet"):
            frames.append(pl.read_parquet(path))
        elif path.endswith(".csv"):
            frames.append(pl.read_csv(path))
        else:
            raise ValueError(f"Unsupported file type: {path}")
    df = pl.concat(frames, how="diagonal")

    if feature_cols is not None:
        cols_to_keep = list(feature_cols) + ([label_col] if label_col else [])
        df = df.select([c for c in cols_to_keep if c in df.columns])

    effective_label_col = label_col

print(f"DataFrame shape: {df.shape}")
if effective_label_col and effective_label_col in df.columns:
    n_anom = df[effective_label_col].sum()
    print(f"Anomaly rate: {n_anom}/{len(df)} = {n_anom/len(df):.3%}")
else:
    print("No ground-truth label column available — evaluation metrics will be skipped.")
    effective_label_col = None

## 2 — EDA

In [ ]:
# ---------------------------------------------------------------------------
# Schema, null counts, summary stats
# ---------------------------------------------------------------------------
print("=== Schema ===")
for col_name, dtype in df.schema.items():
    null_count = df[col_name].null_count()
    print(f"  {col_name:30s}  {str(dtype):20s}  nulls={null_count}")

print("\n=== Summary stats ===")
print(df.describe())

if effective_label_col and effective_label_col in df.columns:
    print("\n=== Class distribution ===")
    vc = df[effective_label_col].value_counts().sort(effective_label_col)
    for row in vc.iter_rows(named=True):
        label_val = row[effective_label_col]
        cnt = row["count"]
        label_name = "anomaly" if label_val == 1 else "inlier"
        print(f"  {label_name} (label={label_val}): {cnt:6d}  ({cnt/len(df):.3%})")

In [ ]:
# ---------------------------------------------------------------------------
# 2D scatter of first two features
# ---------------------------------------------------------------------------
feature_names_all = [c for c in df.columns if c != effective_label_col]
n_features = len(feature_names_all)

if n_features >= 2:
    f0, f1 = feature_names_all[0], feature_names_all[1]
    x0 = df[f0].to_numpy()
    x1 = df[f1].to_numpy()

    if effective_label_col and effective_label_col in df.columns:
        labels_np = df[effective_label_col].to_numpy()
        mask_in = labels_np == 0
        mask_out = labels_np == 1
    else:
        mask_in = np.ones(len(df), dtype=bool)
        mask_out = np.zeros(len(df), dtype=bool)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(x0[mask_in], x1[mask_in], s=10, alpha=0.4, color="steelblue", label="inlier")
    if mask_out.any():
        ax.scatter(x0[mask_out], x1[mask_out], s=20, alpha=0.7, color="crimson", label="anomaly")
    ax.set_xlabel(f0)
    ax.set_ylabel(f1)
    ax.set_title("EDA: first two features (colored by label)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/eda_scatter.png", dpi=120)
    plt.show()
    plt.close()
    print(f"Saved: {plots_dir}/eda_scatter.png")
else:
    print("Only 1 feature — skipping 2D scatter plot.")

In [ ]:
# ---------------------------------------------------------------------------
# Feature distribution histograms (inlier vs anomaly overlay)
# ---------------------------------------------------------------------------
plot_features = feature_names_all[:8]
n_plot = len(plot_features)
ncols = min(4, n_plot)
nrows = (n_plot + ncols - 1) // ncols

if effective_label_col and effective_label_col in df.columns:
    labels_np = df[effective_label_col].to_numpy()
    df_in = df.filter(pl.col(effective_label_col) == 0)
    df_out = df.filter(pl.col(effective_label_col) == 1)
else:
    df_in = df
    df_out = None

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes_flat = np.array(axes).flatten() if n_plot > 1 else [axes]

for i, feat in enumerate(plot_features):
    ax = axes_flat[i]
    vals_in = df_in[feat].drop_nulls().to_numpy()
    ax.hist(vals_in, bins=40, alpha=0.5, color="steelblue", label="inlier", density=True)
    if df_out is not None and len(df_out) > 0:
        vals_out = df_out[feat].drop_nulls().to_numpy()
        ax.hist(vals_out, bins=40, alpha=0.5, color="crimson", label="anomaly", density=True)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

for j in range(n_plot, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Feature Distributions: Inlier vs Anomaly", fontsize=12)
plt.tight_layout()
plt.savefig(f"{plots_dir}/eda_distributions.png", dpi=120)
plt.show()
plt.close()
print(f"Saved: {plots_dir}/eda_distributions.png")

## 3 — Train / Test Split

In [ ]:
# ---------------------------------------------------------------------------
# Train / test split and feature scaling
# ---------------------------------------------------------------------------
feature_names_all = [c for c in df.columns if c != effective_label_col]
X_full = df.select(feature_names_all).to_numpy().astype(np.float64)

if effective_label_col and effective_label_col in df.columns:
    y_full = df[effective_label_col].to_numpy().astype(np.int32)
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y_full, test_size=test_size, random_state=random_state,
        stratify=y_full
    )
else:
    y_full = None
    X_train, X_test = train_test_split(
        X_full, test_size=test_size, random_state=random_state
    )
    y_train = y_test = None

# Scale features — fit only on train set
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train_sc.shape[0]} rows")
print(f"Test:  {X_test_sc.shape[0]} rows")
if y_train is not None:
    print(f"Train anomaly rate: {y_train.mean():.3%}")
    print(f"Test  anomaly rate: {y_test.mean():.3%}")
print("Note: detectors are trained ONLY on train data (unsupervised — no labels used for training).")

## 4 — Anomaly Detectors

In [ ]:
# ---------------------------------------------------------------------------
# Define detectors
# ---------------------------------------------------------------------------
detectors = {
    "isolation_forest": IsolationForest(
        contamination=contamination, random_state=random_state, n_jobs=-1
    ),
    "local_outlier_factor": LocalOutlierFactor(
        contamination=contamination, novelty=True, n_jobs=-1
    ),
    "one_class_svm": OneClassSVM(nu=contamination, kernel="rbf"),
    "elliptic_envelope": EllipticEnvelope(
        contamination=contamination, random_state=random_state
    ),
}

print("Detector configurations:")
for name, det in detectors.items():
    print(f"  {name}: {det}")

In [ ]:
# ---------------------------------------------------------------------------
# Fit all detectors on X_train; predict on X_test
# ---------------------------------------------------------------------------
# Each sklearn detector predict() returns +1 (inlier) or -1 (anomaly).
# We convert to binary: 1 = anomaly, 0 = inlier.
# For scores: we negate decision_function / score_samples so that
# higher score = more anomalous (consistent direction for ROC/PR curves).

predictions = {}   # {name: binary np.array, 1=anomaly}
scores = {}        # {name: float np.array, higher=more anomalous}

for name, det in detectors.items():
    print(f"Fitting {name}...", end=" ", flush=True)
    det.fit(X_train_sc)

    raw_pred = det.predict(X_test_sc)           # +1 or -1
    binary_pred = (raw_pred == -1).astype(np.int32)  # 1 = anomaly
    predictions[name] = binary_pred

    # Anomaly score: negate so higher -> more anomalous
    raw_score = det.score_samples(X_test_sc)    # all sklearn detectors support this
    scores[name] = -raw_score

    print(f"done. Predicted anomalies: {binary_pred.sum()} / {len(binary_pred)}")

print("\nAll detectors trained and predicted.")

## 5 — Evaluation

In [ ]:
# ---------------------------------------------------------------------------
# Compute metrics for each detector
# ---------------------------------------------------------------------------
results = {}

for name in detectors:
    pred = predictions[name]
    score = scores[name]
    row = {"n_predicted_anomalies": int(pred.sum())}

    if y_test is not None:
        row["precision"] = float(precision_score(y_test, pred, zero_division=0))
        row["recall"]    = float(recall_score(y_test, pred, zero_division=0))
        row["f1"]        = float(f1_score(y_test, pred, zero_division=0))
        # ROC-AUC and PR-AUC need score
        if len(np.unique(y_test)) > 1:
            row["roc_auc"] = float(roc_auc_score(y_test, score))
            row["pr_auc"]  = float(average_precision_score(y_test, score))
        else:
            row["roc_auc"] = float("nan")
            row["pr_auc"]  = float("nan")
    else:
        row.update({"precision": float("nan"), "recall": float("nan"),
                    "f1": float("nan"), "roc_auc": float("nan"), "pr_auc": float("nan")})

    results[name] = row

# Print comparison table
print(f"{'Detector':<25} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>9} {'PR-AUC':>8} {'#Anomalies':>11}")
print("-" * 85)
for name, row in results.items():
    print(
        f"{name:<25} "
        f"{row['precision']:>10.4f} "
        f"{row['recall']:>8.4f} "
        f"{row['f1']:>8.4f} "
        f"{row['roc_auc']:>9.4f} "
        f"{row['pr_auc']:>8.4f} "
        f"{row['n_predicted_anomalies']:>11d}"
    )

In [ ]:
# ---------------------------------------------------------------------------
# ROC curves for all 4 detectors
# ---------------------------------------------------------------------------
if y_test is not None and len(np.unique(y_test)) > 1:
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["steelblue", "darkorange", "forestgreen", "crimson"]
    for (name, score), color in zip(scores.items(), colors):
        fpr, tpr, _ = roc_curve(y_test, score)
        auc_val = results[name]["roc_auc"]
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})", color=color, lw=2)
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves — All Detectors")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/roc_curves.png", dpi=120)
    plt.show()
    plt.close()
    print(f"Saved: {plots_dir}/roc_curves.png")
else:
    print("No ground-truth labels — skipping ROC curves.")

In [ ]:
# ---------------------------------------------------------------------------
# Precision-Recall curves for all 4 detectors
# ---------------------------------------------------------------------------
if y_test is not None and len(np.unique(y_test)) > 1:
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["steelblue", "darkorange", "forestgreen", "crimson"]
    for (name, score), color in zip(scores.items(), colors):
        prec, rec, _ = precision_recall_curve(y_test, score)
        pr_auc_val = results[name]["pr_auc"]
        ax.plot(rec, prec, label=f"{name} (PR-AUC={pr_auc_val:.3f})", color=color, lw=2)
    baseline = y_test.mean()
    ax.axhline(baseline, color="k", linestyle="--", lw=1, label=f"random (baseline={baseline:.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curves — All Detectors")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/pr_curves.png", dpi=120)
    plt.show()
    plt.close()
    print(f"Saved: {plots_dir}/pr_curves.png")
else:
    print("No ground-truth labels — skipping PR curves.")

In [ ]:
# ---------------------------------------------------------------------------
# Bar chart: F1, ROC-AUC, PR-AUC comparison
# ---------------------------------------------------------------------------
if y_test is not None:
    det_names = list(results.keys())
    metrics_to_plot = ["f1", "roc_auc", "pr_auc"]
    metric_labels   = ["F1", "ROC-AUC", "PR-AUC"]
    x = np.arange(len(det_names))
    width = 0.25
    colors = ["steelblue", "darkorange", "forestgreen"]

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (metric, label, color) in enumerate(zip(metrics_to_plot, metric_labels, colors)):
        vals = [results[n][metric] for n in det_names]
        bars = ax.bar(x + i * width, vals, width, label=label, color=color, alpha=0.85)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8
                )

    ax.set_xticks(x + width)
    ax.set_xticklabels([n.replace("_", "\n") for n in det_names], fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title("Detector Comparison: F1 / ROC-AUC / PR-AUC")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/metric_comparison.png", dpi=120)
    plt.show()
    plt.close()
    print(f"Saved: {plots_dir}/metric_comparison.png")
else:
    print("No ground-truth labels — skipping metric comparison bar chart.")

In [ ]:
# ---------------------------------------------------------------------------
# Confusion matrices (2x2 grid)
# ---------------------------------------------------------------------------
if y_test is not None:
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    det_list = list(detectors.keys())
    for ax, name in zip(axes.flatten(), det_list):
        pred = predictions[name]
        cm = confusion_matrix(y_test, pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["inlier", "anomaly"])
        disp.plot(ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(name.replace("_", " ").title(), fontsize=10)
    plt.suptitle("Confusion Matrices — All Detectors", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/confusion_matrices.png", dpi=120)
    plt.show()
    plt.close()
    print(f"Saved: {plots_dir}/confusion_matrices.png")
else:
    print("No ground-truth labels — skipping confusion matrices.")

In [ ]:
# ---------------------------------------------------------------------------
# 2D decision boundary visualisation (first 2 scaled features only)
# ---------------------------------------------------------------------------
n_features = X_test_sc.shape[1]

if n_features >= 2:
    x_min = X_test_sc[:, 0].min() - 1.0
    x_max = X_test_sc[:, 0].max() + 1.0
    y_min = X_test_sc[:, 1].min() - 1.0
    y_max = X_test_sc[:, 1].max() + 1.0
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200),
    )
    grid_2d = np.c_[xx.ravel(), yy.ravel()]

    # If features > 2, pad with zeros for the remaining dimensions
    if n_features > 2:
        grid_pad = np.zeros((grid_2d.shape[0], n_features - 2))
        grid_full = np.hstack([grid_2d, grid_pad])
    else:
        grid_full = grid_2d

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    det_list = list(detectors.keys())

    if y_test is not None:
        true_labels = y_test
    else:
        true_labels = np.zeros(len(X_test_sc), dtype=np.int32)

    for ax, name in zip(axes.flatten(), det_list):
        det = detectors[name]
        # score_samples returns higher for inliers; negate for anomaly score
        Z = -det.score_samples(grid_full)
        Z = Z.reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=50, cmap="RdBu_r", alpha=0.6)
        mask_in  = true_labels == 0
        mask_out = true_labels == 1
        ax.scatter(
            X_test_sc[mask_in, 0], X_test_sc[mask_in, 1],
            s=8, color="steelblue", alpha=0.5, label="inlier"
        )
        if mask_out.any():
            ax.scatter(
                X_test_sc[mask_out, 0], X_test_sc[mask_out, 1],
                s=20, color="crimson", alpha=0.8, label="anomaly"
            )
        ax.set_title(name.replace("_", " ").title(), fontsize=10)
        ax.legend(fontsize=7, loc="upper right")
        ax.set_xlabel("feature 0 (scaled)")
        ax.set_ylabel("feature 1 (scaled)")

    plt.suptitle("Decision Boundaries (first 2 features, higher=more anomalous)", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{plots_dir}/decision_boundaries.png", dpi=120)
    plt.show()
    plt.close()
    print(f"Saved: {plots_dir}/decision_boundaries.png")
else:
    print("Only 1 feature — skipping 2D decision boundary plot.")

## 6 — Metrics JSON Output

In [ ]:
# ---------------------------------------------------------------------------
# Write metrics JSON
# ---------------------------------------------------------------------------
run_meta = {
    "timestamp": RUN_START.isoformat(),
    "contamination": contamination,
    "test_size": test_size,
    "random_state": random_state,
    "n_train": int(X_train_sc.shape[0]),
    "n_test": int(X_test_sc.shape[0]),
    "n_features": int(X_test_sc.shape[1]),
    "anomaly_rate_train": float(y_train.mean()) if y_train is not None else None,
    "anomaly_rate_test":  float(y_test.mean())  if y_test  is not None else None,
}

# Sanitise NaN -> None for JSON serialisation
def _clean(v):
    if isinstance(v, float) and np.isnan(v):
        return None
    return v

metrics_out = {
    "run_metadata": run_meta,
    "detectors": {
        name: {k: _clean(v) for k, v in row.items()}
        for name, row in results.items()
    },
}

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, "w") as f:
    json.dump(metrics_out, f, indent=2)

print(f"Metrics JSON saved to: {metrics_json_path}")
print(json.dumps(metrics_out, indent=2))

## 7 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Summary: rank by ROC-AUC, show table, identify best detector
# ---------------------------------------------------------------------------
print("=" * 60)
print("ANOMALY DETECTION — RUN SUMMARY")
print("=" * 60)
print(f"  Timestamp  : {RUN_START.isoformat()}")
print(f"  Detectors  : {', '.join(detectors.keys())}")
print(f"  Train rows : {X_train_sc.shape[0]}")
print(f"  Test rows  : {X_test_sc.shape[0]}")
print(f"  Features   : {X_test_sc.shape[1]}")
print(f"  Contamination: {contamination}")
print()

if y_test is not None:
    # Rank by ROC-AUC (descending)
    ranked = sorted(results.items(), key=lambda kv: kv[1].get("roc_auc", 0), reverse=True)
    print(f"{'Rank':<5} {'Detector':<25} {'ROC-AUC':>9} {'PR-AUC':>8} {'F1':>7} {'Precision':>10} {'Recall':>8}")
    print("-" * 75)
    for rank, (name, row) in enumerate(ranked, 1):
        print(
            f"{rank:<5} {name:<25} "
            f"{row['roc_auc']:>9.4f} "
            f"{row['pr_auc']:>8.4f} "
            f"{row['f1']:>7.4f} "
            f"{row['precision']:>10.4f} "
            f"{row['recall']:>8.4f}"
        )
    best_name = ranked[0][0]
    print(f"\nBest detector by ROC-AUC: {best_name} ({ranked[0][1]['roc_auc']:.4f})")
else:
    print("No ground-truth labels provided — evaluation metrics unavailable.")
    print("Detector predicted anomaly counts:")
    for name, row in results.items():
        print(f"  {name}: {row['n_predicted_anomalies']} anomalies predicted")

print()
print("Output paths:")
print(f"  Metrics JSON : {metrics_json_path}")
print(f"  Plots        : {plots_dir}/")
if executed_notebook_path:
    print(f"  Notebook     : {executed_notebook_path}")
print("=" * 60)